# RAG Pipeline with LangChain
## Notebook 5 — AI Engineer Practical Series

In Notebook 4 we built RAG from scratch — every line manually.
Now we rebuild the same pipeline using LangChain in a fraction of the code.

### What we cover:
1. LangChain document loaders & text splitters
2. LangChain embeddings & vector store (Chroma)
3. LangChain retriever
4. LangChain LLM chain
5. Full RAG pipeline in ~10 lines

In [5]:
# Installs the required dependencies for the project.
%pip install langchain langchain-groq langchain-community chromadb sentence-transformers tf-keras python-dotenv langchain-text-splitters

   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   --- ------------------------------------ 2.1/23.5 MB 13.0 MB/s eta 0:00:02
   ---------- ----------------------------- 6.3/23.5 MB 16.8 MB/s eta 0:00:02
   ------------------ --------------------- 10.7/23.5 MB 18.6 MB/s eta 0:00:01
   ------------------------- -------------- 15.2/23.5 MB 19.5 MB/s eta 0:00:01
   --------------------------------- ------ 19.7/23.5 MB 19.7 MB/s eta 0:00:01
   ---------------------------------------  23.3/23.5 MB 19.7 MB/s eta 0:00:01
   ---------------------------------------- 23.5/23.5 MB 18.6 MB/s  0:00:01
   ---------------------------------------- 0.0/4.6 MB ? eta -:--:--
   ---------------------- ----------------- 2.6/4.6 MB 13.7 MB/s eta 0:00:01
   ---------------------------------------- 4.6/4.6 MB 11.6 MB/s  0:00:00
   ---------------------------------------- 0.0/13.0 MB ? eta -:--:--
   ------- -------------------------------- 2.4/13.0 MB 11.2 MB/s eta 0:00:01
   -----------

  You can safely remove it manually.


In [ ]:
# Imports the necessary libraries and loads environment variables.
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer

load_dotenv()

True

In [ ]:
# Initializes the LLM and embeddings models using the provided API key and model names from environment variables.

# LLM
llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model_name=os.getenv("GROQ_MODEL")
)

# Embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

print("LLM ready ✅")
print("Embeddings ready ✅")

LLM ready ✅
Embeddings ready ✅



## Step 1: Load & Chunk Documents
Split documents into smaller chunks.
In production this would load from PDFs, Word docs, websites.

In [10]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Our knowledge base
raw_docs = [
    "RAG stands for Retrieval-Augmented Generation. It combines information retrieval with text generation to produce accurate, grounded responses.",
    "LangChain is a framework for building LLM applications. It provides tools for chaining LLM calls, memory management, and agent creation.",
    "Vector databases store embeddings — numerical representations of text. They enable semantic search by finding similar vectors using cosine similarity.",
    "Fine-tuning trains a pre-existing model on new data. LoRA is a popular fine-tuning technique that only trains small adapter matrices.",
    "Prompt engineering is the practice of designing inputs to get reliable outputs from LLMs. Key techniques include few-shot prompting and chain of thought.",
    "FastAPI is a modern Python web framework for building APIs. It is the standard for serving ML models in production.",
    "Docker packages applications into containers ensuring consistent runs across environments.",
    "LangSmith is an observability platform for LLM applications providing tracing, evaluation and monitoring.",
]

# Convert to LangChain Document objects
docs = [Document(page_content=d) for d in raw_docs]

# Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = splitter.split_documents(docs)

print(f"Documents: {len(docs)}")
print(f"Chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"[{i}] {chunk.page_content[:80]}...")

Documents: 8
Chunks: 8
[0] RAG stands for Retrieval-Augmented Generation. It combines information retrieval...
[1] LangChain is a framework for building LLM applications. It provides tools for ch...
[2] Vector databases store embeddings — numerical representations of text. They enab...
[3] Fine-tuning trains a pre-existing model on new data. LoRA is a popular fine-tuni...
[4] Prompt engineering is the practice of designing inputs to get reliable outputs f...
[5] FastAPI is a modern Python web framework for building APIs. It is the standard f...
[6] Docker packages applications into containers ensuring consistent runs across env...
[7] LangSmith is an observability platform for LLM applications providing tracing, e...


## Step 2: Embed & Store in Chroma
Chroma is a local vector database — stores embeddings on disk.
No external service needed for development.

In [11]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print(f"Vector store created ✅")
print(f"Documents indexed: {vectorstore._collection.count()}")

Vector store created ✅
Documents indexed: 8


## Step 3: Retriever
LangChain wraps the vector store into a retriever.
One line replaces everything we built manually in Notebook 4.

In [12]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

# Test retrieval
query = "How does RAG work?"
results = retriever.invoke(query)

print(f"Query: {query}\n")
for i, doc in enumerate(results):
    print(f"[{i}] {doc.page_content}")

Query: How does RAG work?

[0] RAG stands for Retrieval-Augmented Generation. It combines information retrieval with text generation to produce accurate, grounded responses.
[1] LangChain is a framework for building LLM applications. It provides tools for chaining LLM calls, memory management, and agent creation.


## Step 4: Full RAG Chain
LangChain connects retriever + prompt + LLM into one pipeline.
Compare this to the 50+ lines we wrote in Notebook 4.

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Prompt template
prompt = ChatPromptTemplate.from_template("""
Answer the question using ONLY the context below.
If the answer isn't in the context say 'Not found in knowledge base.'
Always cite which part of the context you used.

Context: {context}
Question: {question}
""")

# Chain
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain ready ✅")

RAG chain ready ✅


In [14]:
questions = [
    "What is LangChain used for?",
    "How does Docker help developers?",
    "What is the capital of France?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag_chain.invoke(q)}")
    print("─" * 50)

Q: What is LangChain used for?
A: LangChain is a framework for building LLM applications. It provides tools for chaining LLM calls, memory management, and agent creation. (Context)

It is used for building LLM applications.
──────────────────────────────────────────────────
Q: How does Docker help developers?
A: Docker helps developers by packaging applications into containers ensuring consistent runs across environments. (Context: Docker packages applications into containers ensuring consistent runs across environments.)
──────────────────────────────────────────────────
Q: What is the capital of France?
A: Not found in knowledge base.
──────────────────────────────────────────────────
